# Step 0 · Configure a V4 run

**Choose the pathogen, host, run size, and PubMed sample size in the first code cell.** The smoke test selects a small deterministic pathogen set; full mode selects every pathogen in the chosen group or groups and crosses them with the chosen host or hosts. The pathogen–host evidence sample is capped at 200 records per pair, including the 30 earliest PMID IDs. Pair-aware searches use PubMed hit counts for extrapolation; only the selected sample receives abstracts. Full mode never expands the abstract sample beyond this cap.

Run this notebook first. It writes `configs/pilot_v1.yaml` for the later phase notebooks. It also asks the configured LLM for host-mention cues and records them for the PubMed host clause. Every phase writes console progress and a durable log under `data/v4/logs/<run-id>/`.

In [16]:
from pathlib import Path
import yaml
HOST_CHOICES_ROOT = Path.cwd().resolve()
while HOST_CHOICES_ROOT != HOST_CHOICES_ROOT.parent and not (HOST_CHOICES_ROOT / 'configs/host_choices.yaml').is_file():
    HOST_CHOICES_ROOT = HOST_CHOICES_ROOT.parent
HOST_CHOICES_PATH = HOST_CHOICES_ROOT / 'configs/host_choices.yaml'
if not HOST_CHOICES_PATH.is_file():
    raise FileNotFoundError('Could not find configs/host_choices.yaml; open the notebook from the V4 repository')
HOST_NAME_TO_ID = yaml.safe_load(HOST_CHOICES_PATH.read_text(encoding='utf-8'))['host_name_to_id']
print('Host choices from configs/host_choices.yaml:')
for host_name, host_id in HOST_NAME_TO_ID.items():
    print(f'  {host_name}: {host_id}')


Host choices from configs/host_choices.yaml:
  Human: ncbi:9606
  Cattle: ncbi:9913
  Cow: ncbi:9913
  Sheep: ncbi:9940
  Chicken: ncbi:9031
  Pig: ncbi:9823
  Maize: ncbi:4577
  Corn: ncbi:4577
  Rice: ncbi:4530
  Wheat: ncbi:4565
  Soybean: ncbi:3847
  Soy: ncbi:3847
  Atlantic cod: ncbi:8049
  Cod: ncbi:8049
  Atlantic salmon: ncbi:8030
  Salmon: ncbi:8030


## Choose the run before loading the workflow

In [17]:
# Edit these controls first. Host names are loaded from configs/host_choices.yaml and printed above.
RUN_ID = 'human-parasites-2026-11-03'  # independent full Human × Parasites cohort
RUN_MODE = 'full'  # choose 'smoke_test' or 'full'
SELECTED_HOST_NAMES = ['Human']  # e.g. Human, Rice, Atlantic salmon, Atlantic cod
SELECTED_PATHOGEN_GROUPS = ['parasites']  # choose bacteria, fungi, oomycetes, parasites, and/or viruses
SMOKE_PATHOGENS_PER_GROUP = 1  # applies only when RUN_MODE='smoke_test'
SMOKE_PATHOGEN_IDS = []  # optional exact species selection for a one-pair smoke test
MAX_PMIDS_PER_PAIR = 200  # hard maximum abstract-screen sample per pathogen–host pair
OLDEST_PMIDS_PER_PAIR = 30  # lowest PMID IDs included before the random sample
PMID_SAMPLE_SEED = 42  # stable random ordering; keep fixed to extend a sample consistently
SELECTION_SEED = 'human-parasites-2026-11-03'
HOST_SEARCH_EXPANSION_ENABLED = True  # use LLM-proposed host mention cues in PubMed retrieval
HOST_SEARCH_EXPANSION_MODEL = 'gpt-6.1-terra'  # default model for generic host-term expansion
HOST_MESH_TERMS_BY_ID = {'ncbi:9606': ['Humans']}  # controlled vocabulary; text cues are generated separately

if not HOST_SEARCH_EXPANSION_MODEL.strip():
    raise ValueError('HOST_SEARCH_EXPANSION_MODEL must name a model')
if RUN_MODE not in {'smoke_test', 'full'}:
    raise ValueError("RUN_MODE must be 'smoke_test' or 'full'")
if not SELECTED_PATHOGEN_GROUPS or not SELECTED_HOST_NAMES:
    raise ValueError('Select at least one pathogen group and one host')
if SMOKE_PATHOGENS_PER_GROUP < 1:
    raise ValueError('SMOKE_PATHOGENS_PER_GROUP must be positive')
if not isinstance(MAX_PMIDS_PER_PAIR, int) or not 1 <= MAX_PMIDS_PER_PAIR <= 200:
    raise ValueError('MAX_PMIDS_PER_PAIR must be an integer from 1 to 200; all-hit abstract screening is disabled')
if OLDEST_PMIDS_PER_PAIR < 0 or (MAX_PMIDS_PER_PAIR is not None and OLDEST_PMIDS_PER_PAIR > MAX_PMIDS_PER_PAIR):
    raise ValueError('OLDEST_PMIDS_PER_PAIR must be nonnegative and no greater than the sample cap')
print('Mode:', RUN_MODE, '| pathogen groups:', SELECTED_PATHOGEN_GROUPS, '| hosts:', SELECTED_HOST_NAMES)
print('PubMed sample cap:', MAX_PMIDS_PER_PAIR, 'abstracts per pathogen–host pair (maximum 200) | oldest PMID IDs:', OLDEST_PMIDS_PER_PAIR, '| seed:', PMID_SAMPLE_SEED)
print('Host mention expansion:', HOST_SEARCH_EXPANSION_ENABLED, '| model:', HOST_SEARCH_EXPANSION_MODEL)


Mode: full | pathogen groups: ['parasites'] | hosts: ['Human']
PubMed sample cap: 200 abstracts per pathogen–host pair (maximum 200) | oldest PMID IDs: 30 | seed: 42
Host mention expansion: True | model: gpt-6.1-terra


In [18]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT = PROJECT_ROOT
ROOT, LOG, LOG_PATH = configure_notebook('00_configure_run', RUN_ID)
print('Project root:', ROOT)
print('Run ID:', RUN_ID)
print('Session log:', LOG_PATH)

2026-11-03 12:21 INFO Notebook session started at 2026-11-03T12:21:11.914424+00:00
2026-11-03 12:21 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 12:21 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/00_configure_run.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
Run ID: human-parasites-2026-11-03
Session log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/00_configure_run.log


## Step 1 · Load taxonomy and resolve the common-name map

In [19]:
from IPython.display import display
import pandas as pd
import yaml
import json

CONFIG_PATH = ROOT / 'configs/pilot_v1.yaml'
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
STAGING_ROOT = TAXONOMY_ROOT.parent
base_pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
base_aliases = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_aliases.parquet')
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
from pathogen_atlas_v4.pathogen_taxonomy import load_ncbi_pathogen_additions, merge_pathogen_catalogues, PATHOGEN_TAXONOMY_ROOTS
TAXDUMP_ROOT = ROOT / 'data/source_taxonomy/ncbi'
TAXDUMP_MANIFEST = json.loads((TAXDUMP_ROOT / 'source_manifest.json').read_text())
TAXONOMY_RELEASE = f"NCBI Taxonomy {TAXDUMP_MANIFEST['retrieved_utc']} sha256:{TAXDUMP_MANIFEST['archive_sha256']}"
from pathogen_atlas_v4.host_taxonomy.ncbi import load_ncbi_hosts
missing_host_taxa = sorted({host_id.split(':', 1)[1] for host_id in HOST_NAME_TO_ID.values()} - set(hosts.host_id.astype(str).str.split(':').str[-1]))
if missing_host_taxa:
    extra_hosts = load_ncbi_hosts(TAXDUMP_ROOT, missing_host_taxa, release=TAXONOMY_RELEASE)
    hosts = pd.concat([hosts, extra_hosts], ignore_index=True).drop_duplicates('host_id', keep='last')
TAXONOMY_SOURCE_VERSION = f"ncbi-taxdump:{TAXDUMP_MANIFEST['archive_sha256']}"
new_pathogens, new_aliases, scoped_taxa = load_ncbi_pathogen_additions(
    TAXDUMP_ROOT, SELECTED_PATHOGEN_GROUPS, release=TAXONOMY_RELEASE,
    source_version=TAXONOMY_SOURCE_VERSION)
pathogens, aliases = merge_pathogen_catalogues(base_pathogens, base_aliases,
                                                new_pathogens, new_aliases, scoped_taxa)
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
RUN_TAXONOMY_ROOT.mkdir(parents=True, exist_ok=True)
hosts.to_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet', index=False)
pathogens.to_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet', index=False)
aliases.to_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet', index=False)
known_host_ids = set(hosts.host_id.astype(str))
unknown_host_names = sorted(set(SELECTED_HOST_NAMES) - set(HOST_NAME_TO_ID))
if unknown_host_names:
    raise ValueError(f'Unknown common host names: {unknown_host_names}')
HOST_IDS = list(dict.fromkeys(HOST_NAME_TO_ID[name] for name in SELECTED_HOST_NAMES))
unknown_host_ids = sorted(set(HOST_IDS) - known_host_ids)
if unknown_host_ids:
    raise ValueError(f'Host name map contains IDs absent from pinned taxonomy: {unknown_host_ids}')
known_groups = set(pathogens.pathogen_group.astype(str))
unknown_groups = sorted(set(SELECTED_PATHOGEN_GROUPS) - known_groups)
if unknown_groups:
    raise ValueError(f'Unknown pathogen groups: {unknown_groups}')
host_scientific_names = hosts.set_index('host_id').accepted_name.astype(str).to_dict()
host_map = pd.DataFrame([{'common_name': name, 'scientific_name': host_scientific_names[host_id], 'host_id': host_id}
                         for name, host_id in HOST_NAME_TO_ID.items()])
print('Taxonomy source:', TAXONOMY_ROOT)
print('Available host choices:')
display(host_map)
print('Selected host IDs:', HOST_IDS)
LOG.info('Resolved host names=%s to IDs=%s', SELECTED_HOST_NAMES, HOST_IDS)

Taxonomy source: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v5-hosts/taxonomy
Available host choices:


,common_name,scientific_name,host_id
0,Human,Homo sapiens,ncbi:9606
1,Cattle,Bos taurus,ncbi:9913
2,Cow,Bos taurus,ncbi:9913
3,Sheep,Ovis aries,ncbi:9940
4,Chicken,Gallus gallus,ncbi:9031
5,Pig,Sus scrofa,ncbi:9823
6,Maize,Zea mays,ncbi:4577
7,Corn,Zea mays,ncbi:4577
8,Rice,Oryza sativa,ncbi:4530
9,Wheat,Triticum aestivum,ncbi:4565


2026-11-03 12:21 INFO Resolved host names=['Human'] to IDs=['ncbi:9606']


Selected host IDs: ['ncbi:9606']


## Step 2 · Build the smoke-test or full pathogen selection

In [20]:
import hashlib

pathogen_count_by_group = {
    group: int(pathogens.pathogen_group.astype(str).eq(group).sum())
    for group in SELECTED_PATHOGEN_GROUPS
}
selected_pathogen_ids = []
selected_pairs = []
if RUN_MODE == 'smoke_test':
    if SMOKE_PATHOGEN_IDS:
        selected_pathogen_ids = list(dict.fromkeys(map(str, SMOKE_PATHOGEN_IDS)))
        selected_rows = pathogens.loc[pathogens.pathogen_id.astype(str).isin(selected_pathogen_ids)]
        if set(selected_rows.pathogen_id.astype(str)) != set(selected_pathogen_ids):
            raise ValueError('SMOKE_PATHOGEN_IDS contains a pathogen absent from the staged taxonomy')
        if set(selected_rows.pathogen_group.astype(str)) != set(SELECTED_PATHOGEN_GROUPS):
            raise ValueError('Smoke pathogen IDs must cover exactly the selected pathogen groups')
        if selected_rows.groupby('pathogen_group').pathogen_id.nunique().to_dict() != {group: 1 for group in SELECTED_PATHOGEN_GROUPS}:
            raise ValueError('Smoke mode requires exactly one pathogen species per selected group')
    else:
        for group in SELECTED_PATHOGEN_GROUPS:
            candidates = pathogens.loc[pathogens.pathogen_group.astype(str).eq(group), 'pathogen_id'].astype(str).unique()
            chosen = sorted(candidates, key=lambda value: hashlib.sha256(f'{SELECTION_SEED}|{value}'.encode()).hexdigest())[:SMOKE_PATHOGENS_PER_GROUP]
            if len(chosen) != SMOKE_PATHOGENS_PER_GROUP:
                raise ValueError(f'Expected {SMOKE_PATHOGENS_PER_GROUP} pathogens for {group}, found {len(chosen)}')
            selected_pathogen_ids.extend(chosen)
    selected_pairs = [{'pathogen_id': pathogen_id, 'host_id': host_id, 'relationship_context': 'disease'}
                      for pathogen_id in selected_pathogen_ids for host_id in HOST_IDS]
    PAIR_GENERATION = 'explicit_pairs'
    PATHOGEN_COUNT = len(selected_pathogen_ids)
    PAIR_COUNT = len(selected_pairs)
else:
    # Full mode records the selected groups and Cartesian-product rule; Step 3 resolves and saves all IDs before acquisition.
    PAIR_GENERATION = 'cartesian_product'
    PATHOGEN_COUNT = sum(pathogen_count_by_group.values())
    PAIR_COUNT = PATHOGEN_COUNT * len(HOST_IDS)

planned_pathogen_ids = selected_pathogen_ids or pathogens.loc[
    pathogens.pathogen_group.astype(str).isin(SELECTED_PATHOGEN_GROUPS), 'pathogen_id'
].astype(str).unique().tolist()
planned_pair_keys = {(str(pathogen_id), str(host_id))
                     for pathogen_id in planned_pathogen_ids for host_id in HOST_IDS}
existing_resolved_path = ROOT / 'data/v4/runs' / RUN_ID / 'resolved_config.json'
if existing_resolved_path.exists():
    existing_resolved = json.loads(existing_resolved_path.read_text(encoding='utf-8'))
    existing_pair_keys = {(str(pair['pathogen_id']), str(pair['host_id']))
                          for pair in existing_resolved.get('pairs', [])}
    if existing_pair_keys and existing_pair_keys != planned_pair_keys:
        raise RuntimeError(
            f'Run ID {RUN_ID!r} already contains a different resolved pair set. '
            'Choose a new RUN_ID so prior checkpoints and data are not mixed.'
        )

group_summary = pd.DataFrame([{'pathogen_group': group,
    'available': pathogen_count_by_group[group],
    'selected': min(pathogen_count_by_group[group], SMOKE_PATHOGENS_PER_GROUP) if RUN_MODE == 'smoke_test' else pathogen_count_by_group[group]}
    for group in SELECTED_PATHOGEN_GROUPS])
display(group_summary)
print(f'{RUN_MODE}: {PATHOGEN_COUNT:,} pathogens × {len(HOST_IDS)} hosts → {PAIR_COUNT:,} candidate pairs')
LOG.info('Selection mode=%s groups=%s pathogens=%d hosts=%d pairs=%d', RUN_MODE, SELECTED_PATHOGEN_GROUPS, PATHOGEN_COUNT, len(HOST_IDS), PAIR_COUNT)

,pathogen_group,available,selected
0,parasites,57151,57151


2026-11-03 12:21 INFO Selection mode=full groups=['parasites'] pathogens=57151 hosts=1 pairs=57151


full: 57,151 pathogens × 1 hosts → 57,151 candidate pairs


## Step 3 · Define taxonomy, query, decision, feature, and logging versions

In [21]:
PROJECT_SETTINGS = {'id': 'pathogen-atlas-v4', 'schema_version': 1}
RUN_SETTINGS = {'run_id': RUN_ID, 'config_version': 'pilot-config-v2',
                'selection_mode': RUN_MODE, 'selection_seed': SELECTION_SEED,
                'host_ids': HOST_IDS, 'pathogen_ids': selected_pathogen_ids,
                'pair_generation': PAIR_GENERATION}
if RUN_MODE == 'smoke_test':
    RUN_SETTINGS['pairs'] = selected_pairs
else:
    RUN_SETTINGS['pathogen_groups'] = SELECTED_PATHOGEN_GROUPS
TAXONOMY_SETTINGS = {'host_source': 'ncbi_taxonomy',
                     'staging_snapshot': STAGING_ROOT.name,
                     'host_release_manifest': 'data/source_taxonomy/ncbi/source_manifest.json',
                     'pathogen_taxonomy_release': TAXONOMY_RELEASE,
                     'pathogen_group_taxon_roots': {group: list(PATHOGEN_TAXONOMY_ROOTS[group])
                                                    for group in SELECTED_PATHOGEN_GROUPS if group in PATHOGEN_TAXONOMY_ROOTS}}
FEATURE_SETTINGS = {'catalogue': 'configs/features.yaml', 'profile_version': yaml.safe_load((ROOT / 'configs/features.yaml').read_text())['profile_version']}
from openai import OpenAI
from pathogen_atlas_v4.host_mentions import (
    DEFAULT_HOST_EXPANSION_MODEL, HOST_EXPANSION_PROMPT_VERSION,
    expand_host_mentions, host_expansion_hash,
)
if HOST_SEARCH_EXPANSION_MODEL != DEFAULT_HOST_EXPANSION_MODEL:
    print('Using host expansion model override:', HOST_SEARCH_EXPANSION_MODEL)
HOST_ROWS = hosts.set_index('host_id')
HOST_SEARCH_EXPANSIONS = {}
if HOST_SEARCH_EXPANSION_ENABLED:
    expansion_client = OpenAI(max_retries=5, timeout=120.0)
    for host_id in HOST_IDS:
        host = HOST_ROWS.loc[host_id]
        host_aliases = host.get('aliases', [])
        if hasattr(host_aliases, 'tolist'):
            host_aliases = host_aliases.tolist()
        common_label = next((name for name in SELECTED_HOST_NAMES if HOST_NAME_TO_ID[name] == host_id), str(host.accepted_name))
        expansion = expand_host_mentions(
            host_id=host_id, accepted_name=str(host.accepted_name), aliases=list(host_aliases or []),
            common_label=common_label, model=HOST_SEARCH_EXPANSION_MODEL, client=expansion_client)
        expansion['mesh_terms'] = list(HOST_MESH_TERMS_BY_ID.get(host_id, []))
        HOST_SEARCH_EXPANSIONS[host_id] = expansion
else:
    for host_id in HOST_IDS:
        host = HOST_ROWS.loc[host_id]
        HOST_SEARCH_EXPANSIONS[host_id] = {
            'host_id': host_id, 'host_label': str(host.accepted_name),
            'accepted_name': str(host.accepted_name), 'model': HOST_SEARCH_EXPANSION_MODEL,
            'prompt_version': HOST_EXPANSION_PROMPT_VERSION, 'prompt_sha256': '', 'terms': [],
            'mesh_terms': list(HOST_MESH_TERMS_BY_ID.get(host_id, [])),
        }
HOST_EXPANSION_HASH = host_expansion_hash(HOST_SEARCH_EXPANSIONS)
HOST_TERM_REVIEW = pd.DataFrame([
    {'host_id': host_id, 'term': item['term'], 'category': item['category'],
     'precision_risk': item['precision_risk'], 'rationale': item['rationale']}
    for host_id, expansion in HOST_SEARCH_EXPANSIONS.items() for item in expansion.get('terms', [])
])
print('Host terms are retrieval cues: quoted Title/Abstract alternatives ORed with host names, then ANDed with pathogen names.')
print('Configured host MeSH terms:', {host_id: row.get('mesh_terms', []) for host_id, row in HOST_SEARCH_EXPANSIONS.items()})
display(HOST_TERM_REVIEW)
ACQUISITION_SETTINGS = {'query_strategy_version': 'pathogen-host-batched-expanded-host-mentions-mesh-title-abstract-v4',
                       'pathogen_literature_query_version': 'pathogen-vetted-aliases-title-abstract-v2',
                       'host_expansion_enabled': HOST_SEARCH_EXPANSION_ENABLED,
                       'host_expansion_model': HOST_SEARCH_EXPANSION_MODEL,
                       'host_expansion_prompt_version': HOST_EXPANSION_PROMPT_VERSION,
                       'host_expansion_hash': HOST_EXPANSION_HASH,
                       'host_search_expansions': HOST_SEARCH_EXPANSIONS,
                       'relationship_context': 'disease',
                       'sampling': {'max_pmids_per_pair': MAX_PMIDS_PER_PAIR,
                                    'oldest_pmids_per_pair': OLDEST_PMIDS_PER_PAIR,
                                    'random_seed': PMID_SAMPLE_SEED,
                                    'sample_policy_version': 'stable-prefix-hash-v1'}}
DECISION_SETTINGS = {'model_version': 'typesafe/jev-1.14', 'screen_pathogen_pairs': RUN_MODE == 'full',
                     'prompt_catalogue': 'configs/prompts.yaml',
                     'prompt_set_version': 'system1-prompt-set-v2',
                     'pathogen_screen_question_version': 'pathogen-screen-system1-v2-threshold-0.7',
                     'pathogen_screen_prompt_version': 'pathogen-screen-noul-v1',
                     'pair_feature_prompt_version': 'pair-feature-noul-v1',
                     'gate_question_version': 'pathogen-screen-system1-v2-threshold-0.7',
                     'outcomes': ['supported','unsupported']}
LOGGING_SETTINGS = {'progress_every': 25, 'save_checkpoint_every': 100, 'max_attempts': 2, 'decision_workers': 1024, 'decision_checkpoint_every': 1000}
CONFIG = {'project': PROJECT_SETTINGS, 'run': RUN_SETTINGS, 'taxonomy': TAXONOMY_SETTINGS,
          'features': FEATURE_SETTINGS, 'acquisition': ACQUISITION_SETTINGS,
          'decisions': DECISION_SETTINGS, 'logging': LOGGING_SETTINGS}
print('Mode:', RUN_MODE, '| pair generation:', PAIR_GENERATION)
print('Selection:', len(selected_pathogen_ids), 'pinned IDs,', len(RUN_SETTINGS.get('pathogen_groups', [])), 'full groups,', PAIR_COUNT, 'candidate pairs')
LOG.info('Built %s run configuration', RUN_MODE)

Host terms are retrieval cues: quoted Title/Abstract alternatives ORed with host names, then ANDed with pathogen names.
Configured host MeSH terms: {'ncbi:9606': ['Humans']}


,host_id,term,category,precision_risk,rationale
0,ncbi:9606,humans,common_name,low,Common plural host designation.
1,ncbi:9606,person,person_or_patient_descriptor,medium,May identify an individual human case.
2,ncbi:9606,persons,person_or_patient_descriptor,medium,Common descriptor for human populations.
3,ncbi:9606,people,person_or_patient_descriptor,medium,Common descriptor for human populations.
4,ncbi:9606,patient,person_or_patient_descriptor,high,May capture clinical cases but can also refer ...
5,ncbi:9606,patients,person_or_patient_descriptor,high,May capture clinical cohorts but can also refe...
6,ncbi:9606,child,age_or_sex_descriptor,medium,Common descriptor for a pediatric case.
7,ncbi:9606,children,age_or_sex_descriptor,medium,Common descriptor for pediatric populations.
8,ncbi:9606,pediatric,age_or_sex_descriptor,medium,Common clinical descriptor for children.
9,ncbi:9606,paediatric,age_or_sex_descriptor,medium,Alternative spelling of pediatric.


2026-11-03 12:21 INFO Built full run configuration


Mode: full | pair generation: cartesian_product
Selection: 0 pinned IDs, 1 full groups, 57151 candidate pairs


## Step 4 · Review the exact pair plan before writing

In [22]:
if RUN_MODE == 'smoke_test':
    pair_review = pd.DataFrame(selected_pairs).merge(
        pathogens[['pathogen_id','pathogen_group','accepted_name']], on='pathogen_id', how='left'
    ).merge(hosts[['host_id','accepted_name']].rename(columns={'accepted_name':'host_name'}), on='host_id', how='left')
    assert pair_review.pathogen_group.notna().all() and pair_review.host_name.notna().all()
    assert len(pair_review) == PAIR_COUNT
    display(pair_review)
else:
    full_plan = group_summary.assign(selected_hosts=len(HOST_IDS),
        candidate_pairs=group_summary.selected * len(HOST_IDS))
    display(full_plan)
    print('Full mode uses a recorded Cartesian-product rule; Step 3 resolves and saves the full ID/pair list before acquisition.')
print('Candidate pairs:', f'{PAIR_COUNT:,}')
LOG.info('Reviewed pair plan for mode=%s, pairs=%d', RUN_MODE, PAIR_COUNT)

,pathogen_group,available,selected,selected_hosts,candidate_pairs
0,parasites,57151,57151,1,57151


2026-11-03 12:21 INFO Reviewed pair plan for mode=full, pairs=57151


Full mode uses a recorded Cartesian-product rule; Step 3 resolves and saves the full ID/pair list before acquisition.
Candidate pairs: 57,151


## Step 5 · Write the shared configuration for later notebooks

In [23]:
SAVE_CONFIGURATION = True
if SAVE_CONFIGURATION:
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Wrote shared configuration:', CONFIG_PATH)
    LOG.info('Wrote shared config to %s', CONFIG_PATH)
else:
    print('No config written; set SAVE_CONFIGURATION=True to save it.')
    LOG.info('Config write skipped')
CONFIG_SUMMARY = pd.DataFrame([
    {'setting':'mode','value':RUN_MODE}, {'setting':'groups','value':', '.join(SELECTED_PATHOGEN_GROUPS)},
    {'setting':'hosts','value':', '.join(SELECTED_HOST_NAMES)}, {'setting':'pathogens','value':f'{PATHOGEN_COUNT:,}'},
    {'setting':'candidate pairs','value':f'{PAIR_COUNT:,}'},
    {'setting':'PMID sample cap per pair','value':MAX_PMIDS_PER_PAIR},
    {'setting':'earliest PMID IDs','value':OLDEST_PMIDS_PER_PAIR},
    {'setting':'PMID sample seed','value':PMID_SAMPLE_SEED}, {'setting':'pair rule','value':PAIR_GENERATION},
    {'setting':'host expansion enabled','value':HOST_SEARCH_EXPANSION_ENABLED},
    {'setting':'host expansion model','value':HOST_SEARCH_EXPANSION_MODEL},
    {'setting':'host cue count','value':sum(len(item.get('terms', [])) for item in HOST_SEARCH_EXPANSIONS.values())},
    {'setting':'host expansion hash','value':HOST_EXPANSION_HASH},
    {'setting':'run ID','value':RUN_ID}, {'setting':'config path','value':str(CONFIG_PATH)},
])
display(CONFIG_SUMMARY)

2026-11-03 12:21 INFO Wrote shared config to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml


Wrote shared configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml


,setting,value
0,mode,full
1,groups,parasites
2,hosts,Human
3,pathogens,"57,151"
4,candidate pairs,"57,151"
5,PMID sample cap per pair,200
6,earliest PMID IDs,30
7,PMID sample seed,42
8,pair rule,cartesian_product
9,host expansion enabled,True


## Step 6 · Save resolved-run paths and execution log locations

In [24]:
import json
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
PATHS = {'run_dir': str(RUN_DIR), 'resolved_config': str(RUN_DIR / 'resolved_config.json'),
         'decision_ledger': str(RUN_DIR / 'decision_ledger.parquet'),
         'raw_responses': str(RUN_DIR / 'raw_responses.jsonl'),
         'feature_profiles': str(RUN_DIR / 'feature_profiles'),
         'log_dir': str(ROOT / 'data/v4/logs' / RUN_ID)}
RUN_RECORD = {'config_path': str(CONFIG_PATH.relative_to(ROOT)), 'config': CONFIG, 'paths': PATHS}
CONFIG_OUTPUT = RUN_DIR / 'notebook_run_config.json'
CONFIG_OUTPUT.write_text(json.dumps(RUN_RECORD, indent=2, ensure_ascii=False) + '\n')
print('Run record:', CONFIG_OUTPUT)
print('Logs:', PATHS['log_dir'])
LOG.info('Saved run record to %s', CONFIG_OUTPUT)

2026-11-03 12:21 INFO Saved run record to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/notebook_run_config.json


Run record: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/notebook_run_config.json
Logs: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03


## Step 7 · Phase sequence and run controls

In [25]:
WORKFLOW_STEPS = [
    {'step':'00 · Configure','preflight':'selected groups, hosts, sample cap, and host mention expansion','outputs':'shared config, Terra-generated host cues, MeSH terms, and expansion hash'},
    {'step':'01 · Taxonomy','preflight':'source releases and taxonomy IDs','outputs':'normalized hosts and pathogens'},
    {'step':'02 · Preview Jev questions','preflight':'selected pair and feature profile','outputs':'typed yes/no question previews; no model calls'},
    {'step':'03 · Acquire PubMed records','preflight':'resolved pair count, pathogen aliases, host mention cues, MeSH, and PubMed access','outputs':'pair searches, hit counts, PMID matches, sampled abstracts'},
    {'step':'04 · Screen pathogen–host pairs','preflight':'sample cap, Jev credentials, and checkpoint coverage','outputs':'completed binary decisions and accepted pairs'},
    {'step':'05 · Estimate publications','preflight':'accepted-pair coverage and complete sample screening','outputs':'sample-based publication estimates and normalized scores'},
    {'step':'06 · Handoff disease evidence','preflight':'complete Step 4 decision ledger','outputs':'pair decisions linked to PMIDs and abstract hashes'},
    {'step':'07 · Resolve feature profiles','preflight':'ordered module resolution and applicability','outputs':'resolved profiles, applicability, and profile hashes'},
    {'step':'08 · Score feature questions','preflight':'accepted pair abstracts and complete feature tasks','outputs':'yes/no trait scores for every applicable question'},
    {'step':'09 · Validate release','preflight':'foreign keys, completeness, Jev provenance','outputs':'pair-aware validation results; release build remains separate'},
    {'step':'10 · Build and inspect Explorer','preflight':'validated pair data, complete features, and eligible IMSTE cohorts','outputs':'additive Explorer database, evidence, features, and host/group coordinates'},
    {'step':'11 · Resolve Wikipedia references','preflight':'resolved run taxonomy','outputs':'Wikipedia references'},
    {'step':'12 · Resolve GBIF references','preflight':'resolved run taxonomy','outputs':'GBIF references'},
]
display(pd.DataFrame(WORKFLOW_STEPS))
print('Mode:', RUN_MODE, '| groups:', SELECTED_PATHOGEN_GROUPS, '| hosts:', SELECTED_HOST_NAMES)
print('Run ID:', RUN_ID, '| candidate pairs:', f'{PAIR_COUNT:,}')
print('Shared config:', CONFIG_PATH, '| run log:', LOG_PATH)
LOG.info('Step 0 complete')

,step,preflight,outputs
0,00 · Configure,"selected groups, hosts, sample cap, and host m...","shared config, Terra-generated host cues, MeSH..."
1,01 · Taxonomy,source releases and taxonomy IDs,normalized hosts and pathogens
2,02 · Preview Jev questions,selected pair and feature profile,typed yes/no question previews; no model calls
3,03 · Acquire PubMed records,"resolved pair count, pathogen aliases, host me...","pair searches, hit counts, PMID matches, sampl..."
4,04 · Screen pathogen–host pairs,"sample cap, Jev credentials, and checkpoint co...",completed binary decisions and accepted pairs
5,05 · Estimate publications,accepted-pair coverage and complete sample scr...,sample-based publication estimates and normali...
6,06 · Handoff disease evidence,complete Step 4 decision ledger,pair decisions linked to PMIDs and abstract ha...
7,07 · Resolve feature profiles,ordered module resolution and applicability,"resolved profiles, applicability, and profile ..."
8,08 · Score feature questions,accepted pair abstracts and complete feature t...,yes/no trait scores for every applicable question
9,09 · Validate release,"foreign keys, completeness, Jev provenance",pair-aware validation results; release build r...


2026-11-03 12:21 INFO Step 0 complete


Mode: full | groups: ['parasites'] | hosts: ['Human']
Run ID: human-parasites-2026-11-03 | candidate pairs: 57,151
Shared config: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml | run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/00_configure_run.log
